# Getting Started with `torchgfn`

This notebook walks you through:

1. What GFlowNets are and why they matter
2. The key concepts you need to know
3. A working example: training a Flow Matching GFlowNet on a 2D grid

If you're already familiar with GFlowNets, skip to **Part 3**.

## 1. GFlowNets in a Few Words

GFlowNets are a family of methods for building *generative models*. We want to be able to *sample* new objects from a given distribution.

The key idea is that we **sequentially construct** objects step by step, training neural networks so that the probability of generating any object is **proportional to its reward**.

The procedure is:

1. Start from an initial state (empty construction)
2. A neural network outputs a probability distribution over possible next actions
3. Sample an action and apply it to get the next state
4. Repeat until a special "exit" action is sampled
5. Evaluate the resulting object with a reward function and update the network

This sounds like Reinforcement Learning, and to some extent it *is*. The key difference: a GFlowNet is **not** trained to maximize reward. It is trained to **sample proportionally to it**.

Objects with reward $r$ are twice as likely to be generated as objects with reward $r/2$. This ensures **diversity**: a converged GFlowNet samples *all modes* of the distribution, proportionally to their value.

## 2. Key Concepts

- **Reward function**: Takes an object and returns a non-negative score. Higher is better.
- **State**: A (possibly incomplete) representation of the object being built. There is an *initial* state $s_0$ we start from, *intermediate* states, and *terminal* states where construction is complete.
- **Action**: A step that transitions one state to another.
- **Trajectory**: The full sequence of states and actions from $s_0$ to a terminal state.
- **DAG**: The space of all possible trajectories forms a directed acyclic graph. GFlowNets require that no action creates a cycle.
- **Forward policy**: The probability distribution over actions at each state (what the neural network learns).
- **Backward policy**: The probability of each action that *could have led to* the current state (needed by some losses).
- **Environment**: Defines the state space, action space, transitions, and reward function.

### The Flow Matching Loss

There are several ways to train GFlowNets. In this notebook we use **Flow Matching** (FM), the simplest objective.

FM treats each edge in the DAG as carrying a "flow" (think of water flowing through a network). The loss enforces two constraints:

1. **Flow conservation**: For every intermediate state, the total incoming flow equals the total outgoing flow.
2. **Reward matching**: For every terminal state, the flow into the exit action equals the reward.

The neural network learns to estimate the log-flow on each edge. Once trained, these flows define a valid sampling policy.

## 3. A Working Example: HyperGrid

We'll train a GFlowNet to sample points on a 2D grid of size $8 \times 8$. The reward function places peaks in the corners, and we want our model to learn to sample proportionally to this reward.

### 3.1 Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from gfn.gym import HyperGrid
from gfn.gflownet import FMGFlowNet
from gfn.estimators import DiscretePolicyEstimator
from gfn.preprocessors import KHotPreprocessor
from gfn.utils.modules import MLP

### 3.2 Environment

The `HyperGrid` environment is a discrete grid where:
- The agent starts at the origin $(0, 0)$
- At each step it can move +1 along any dimension, or take the exit action
- The reward function assigns higher values to the corners of the grid

In [ ]:
ndim = 2
height = 8

env = HyperGrid(ndim=ndim, height=height)

print(f"State space: {height}x{height} = {height**ndim} states")
print(f"Actions: {env.n_actions} (move along each of {ndim} dims + exit)")

Let's visualize the reward landscape:

In [ ]:
# Build a grid of all possible terminal states and evaluate the reward.
grid = torch.stack(
    torch.meshgrid(torch.arange(height), torch.arange(height), indexing="ij"), dim=-1
).reshape(-1, 2)

all_states = env.states_from_tensor(grid.float())
rewards = env.reward(all_states).reshape(height, height)

plt.figure(figsize=(5, 4))
plt.imshow(rewards.numpy(), origin="lower", cmap="viridis")
plt.colorbar(label="Reward")
plt.xlabel("Dim 0")
plt.ylabel("Dim 1")
plt.title("HyperGrid Reward Landscape")
plt.tight_layout()
plt.show()

### 3.3 Model

For Flow Matching, we need a single neural network that estimates the **log edge flow** $\log F(s \to s')$ for each state-action pair.

In `torchgfn`, this is wrapped in a `DiscretePolicyEstimator` that:
1. Preprocesses states (here, using a k-hot encoding)
2. Passes them through an MLP
3. Outputs one logit per action (including exit)

In [ ]:
# Preprocessor: converts discrete grid coordinates to a k-hot vector.
preprocessor = KHotPreprocessor(height=height, ndim=ndim)
print(f"Preprocessor output dim: {preprocessor.output_dim}")

# Neural network: maps preprocessed states to log edge flows.
module = MLP(
    input_dim=preprocessor.output_dim,  # height * ndim = 16
    output_dim=env.n_actions,            # ndim + 1 = 3
    hidden_dim=256,
    n_hidden_layers=2,
)

# Wrap in an estimator.
logF_estimator = DiscretePolicyEstimator(
    module=module,
    n_actions=env.n_actions,
    preprocessor=preprocessor,
)

# Build the FM GFlowNet.
gflownet = FMGFlowNet(logF=logF_estimator)

### 3.4 Training Loop

Each iteration:
1. **Sample** trajectories from the current policy
2. **Convert** them to a `StatesContainer` (the format FM loss expects)
3. **Compute** the FM loss and backpropagate

In [ ]:
optimizer = torch.optim.Adam(gflownet.parameters(), lr=1e-3)

n_iterations = 1000
batch_size = 256
losses = []

for i in range(n_iterations):
    # 1. Sample trajectories from the current policy.
    trajectories = gflownet.sample_trajectories(env, n=batch_size)

    # 2. Convert trajectories to the training format for FM.
    training_samples = gflownet.to_training_samples(trajectories)

    # 3. Compute loss and update.
    optimizer.zero_grad()
    loss = gflownet.loss(env, training_samples)
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if (i + 1) % 250 == 0:
        print(f"Iteration {i + 1}/{n_iterations} | Loss: {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("FM Loss")
plt.title("Training Loss")
plt.yscale("log")
plt.tight_layout()
plt.show()

### 3.5 Evaluating the Learned Policy

If training went well, the GFlowNet should sample terminal states proportionally to the reward. Let's check by sampling many trajectories and comparing the empirical distribution to the true reward.

In [ ]:
# Sample a large batch of trajectories.
n_eval = 10_000
with torch.no_grad():
    eval_trajectories = gflownet.sample_trajectories(env, n=n_eval)

# Extract terminal states (the last real state before exit).
terminal_states = eval_trajectories.terminating_states.tensor.long()

# Build an empirical count over the grid.
empirical_counts = torch.zeros(height, height)
for s in terminal_states:
    empirical_counts[s[0], s[1]] += 1

empirical_dist = empirical_counts / empirical_counts.sum()

# True distribution: reward normalized to a probability.
true_dist = rewards / rewards.sum()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(true_dist.numpy(), origin="lower", cmap="viridis")
axes[0].set_title("True Distribution (Reward)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(empirical_dist.numpy(), origin="lower", cmap="viridis")
axes[1].set_title(f"Learned Distribution ({n_eval} samples)")
plt.colorbar(im1, ax=axes[1])

# Absolute difference.
diff = (empirical_dist - true_dist).abs()
im2 = axes[2].imshow(diff.numpy(), origin="lower", cmap="Reds")
axes[2].set_title("Absolute Difference")
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_xlabel("Dim 0")
    ax.set_ylabel("Dim 1")

plt.tight_layout()
plt.show()

# L1 distance between distributions.
l1 = (empirical_dist - true_dist).abs().sum().item()
print(f"L1 distance between empirical and true distributions: {l1:.4f}")

### 3.6 Estimating the Partition Function

A nice property of Flow Matching: the sum of outgoing flows from the initial state $s_0$ estimates the partition function $Z = \sum_x R(x)$.

In [ ]:
# Evaluate log edge flows at s0.
s0_states = env.states_from_batch_shape((1,))  # A batch containing just s0.

with torch.no_grad():
    log_flows_at_s0 = logF_estimator(s0_states)

estimated_Z = log_flows_at_s0.logsumexp(dim=-1).exp().item()
true_Z = rewards.sum().item()

print(f"Estimated Z: {estimated_Z:.2f}")
print(f"True Z:      {true_Z:.2f}")

## Summary

In this notebook we trained a Flow Matching GFlowNet on a 2D grid using `torchgfn`. The key components were:

| Component | `torchgfn` class | Role |
|-----------|-----------------|------|
| Environment | `HyperGrid` | Defines states, actions, transitions, and rewards |
| Preprocessor | `KHotPreprocessor` | Converts grid coordinates to neural network input |
| Neural network | `MLP` | Estimates log edge flows |
| Estimator | `DiscretePolicyEstimator` | Wraps the network with preprocessing and action handling |
| GFlowNet | `FMGFlowNet` | Implements sampling and the FM loss |

### Next Steps

- Try other losses: `TBGFlowNet` (Trajectory Balance), `DBGFlowNet` (Detailed Balance), `SubTBGFlowNet` (Sub-Trajectory Balance)
- Explore higher-dimensional grids (`ndim=4`, `height=16`)
- Add a replay buffer for off-policy training
- See the other notebooks in `tutorials/notebooks/` for continuous environments, graphs, and more
- Read the [GFlowNet Foundations paper](https://arxiv.org/abs/2111.09266) for the theory